# Course 2: Privacy-Preserving Machine Learning
## Session 1: What models leak

**Goal:** run a membership inference attack against an overfit classifier and see how the leak shrinks when the model overfits less.

**How to use this notebook:** run the cells from top to bottom. Cells marked `TODO` are for you to complete. The lab guide, quiz and assignment for this session are in the course document.

**Testing note:** every cell without a NOT RUN label was run once on a CPU (November 2026) on small built-in or synthetic data. Cells labelled NOT RUN need a library, a dataset download or a GPU; run them on Kaggle or Colab.

### 1. Data and a model that memorizes
Members are the training points. Non-members come from the same source but were never seen. As in real records, one fifth of the recorded labels are wrong, for members and non-members alike. A model can only fit a wrong label by memorizing it.

In [ ]:
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F
from sklearn.datasets import load_digits
from sklearn.metrics import roc_auc_score, roc_curve
torch.manual_seed(0); rng = np.random.default_rng(0)
d = load_digits(); idx = rng.permutation(len(d.data))
X = torch.tensor(d.data / 16.0, dtype=torch.float32); y_true = torch.tensor(d.target)
wrong = torch.tensor(rng.random(len(X)) < 0.2)                       # which records carry a wrong label
y = torch.where(wrong, (y_true + torch.tensor(rng.integers(1, 10, len(X)))) % 10, y_true)   # the recorded labels
mem, non, fresh = idx[:400], idx[400:800], idx[800:]                 # 400 members, 400 non-members, the rest for testing

def train(weight_decay, epochs):
    torch.manual_seed(0)
    net = nn.Sequential(nn.Linear(64, 256), nn.ReLU(), nn.Linear(256, 10))
    opt = torch.optim.Adam(net.parameters(), lr=3e-3, weight_decay=weight_decay)
    for _ in range(epochs):
        loss = F.cross_entropy(net(X[mem]), y[mem]); opt.zero_grad(); loss.backward(); opt.step()
    return net

overfit = train(weight_decay=0.0, epochs=600)

### 2. The attack: threshold the loss
A model is usually more confident on the points it trained on. The attacker computes the loss of each candidate record and guesses 'member' when it is low.

In [ ]:
def losses(net, rows):
    with torch.no_grad(): return F.cross_entropy(net(X[rows]), y[rows], reduction="none").numpy()

def attack(net):
    lm, ln = losses(net, mem), losses(net, non)
    labels = np.r_[np.ones(400), np.zeros(400)]; scores = -np.r_[lm, ln]
    fpr, tpr, _ = roc_curve(labels, scores)
    with torch.no_grad(): test_acc = (net(X[fresh]).argmax(1) == y_true[fresh]).float().mean().item()
    return dict(member_loss=lm.mean(), non_member_loss=ln.mean(), test_acc=test_acc,
                attack_auc=roc_auc_score(labels, scores), tpr_at_5pct_fpr=tpr[fpr <= 0.05].max())

r_over = attack(overfit)
for k, v in r_over.items(): print(f"{k:16s} {v:.3f}")
assert r_over["attack_auc"] > 0.7

An AUC of 0.5 means the attacker is guessing. Also read the last line: the share of members caught while wrongly accusing only 5% of non-members. Recent papers ask for that kind of low-false-positive number, usually at 0.1% or 1%, which needs far more records than we have here.

### 3. Less overfitting, less leakage

In [ ]:
regular = train(weight_decay=0.05, epochs=60)
r_reg = attack(regular)
print(f"{'':16s} overfit   regularized")
for k in r_over: print(f"{k:16s} {r_over[k]:7.3f}   {r_reg[k]:7.3f}")
assert r_reg["attack_auc"] < r_over["attack_auc"] and r_reg["test_acc"] > r_over["test_acc"]

The regularized model is both more accurate and more private. Overfitting is the common cause of both problems, though the two do not always move together.

### 4. Who is most exposed?
Records with a wrong label are outliers. Run the attack separately on outliers and on ordinary records.

In [ ]:
lm, ln = losses(overfit, mem), losses(overfit, non)
wm, wn = wrong[mem].numpy(), wrong[non].numpy()
auc = lambda a, b: roc_auc_score(np.r_[np.ones(len(a)), np.zeros(len(b))], -np.r_[a, b])
auc_clean, auc_outlier = auc(lm[~wm], ln[~wn]), auc(lm[wm], ln[wn])
print(f"attack AUC on ordinary records: {auc_clean:.3f}")
print(f"attack AUC on outlier records : {auc_outlier:.3f}")
assert auc_outlier > auc_clean

Average numbers hide this. The people with rare conditions or unusual records are the ones a model memorizes, and they are often the ones with the most to lose.

### TODO
1. Plot the two loss histograms (members and non-members) for both models.
2. Train shadow models: 8 models on random halves of the data. For each record, compare its loss when it was in training with its loss when it was out. This is the idea behind the stronger attack in session 4.
3. Write three sentences: what would this attack reveal if the training set were the patients of one clinic?